In [4]:
import pathlib
import os 
import pandas as pd

train = "../Dataset/CXRAY/Landmarks_3_10/train.txt"
val = "../Dataset/CXRAY/Landmarks_3_10/val.txt"

train = pd.read_csv(train, header=None)
val = pd.read_csv(val, header=None)

# merge
train = pd.concat([train, val], ignore_index=True)

train

image_base = "../Dataset/CXRAY/Landmarks_3_10/images"
mask_base = "../Dataset/CXRAY/Landmarks_3_10/masks"

i = 1

image_reference = []

nnUNet_image_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset123_CXRAY/imagesTr"
nnUNet_mask_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset123_CXRAY/labelsTr"

os.makedirs(nnUNet_image_base, exist_ok=True)
os.makedirs(nnUNet_mask_base, exist_ok=True)

for row in train.iterrows():
    image_path = os.path.join(image_base, row[1][0])
    mask_path = os.path.join(mask_base, row[1][0])

    image_new_name = "image_%s_0000.png" % i
    mask_new_name = "image_%s.png" % i

    image_reference.append("image_%s" % i)

    # copy image and mask
    image_dest = os.path.join(nnUNet_image_base, image_new_name)
    mask_dest = os.path.join(nnUNet_mask_base, mask_new_name)

    os.system("cp %s %s" % (image_path, image_dest))
    os.system("cp %s %s" % (mask_path, mask_dest))

    i += 1

train = train.assign(image_reference=image_reference)
train.to_csv("../cxray_train_with_reference.csv", index=False)

In [8]:
import pathlib
import os 
import pandas as pd

test = "../Dataset/CXRAY/Landmarks_3_10/test.txt"
test = pd.read_csv(test, header=None)

image_base = "../Dataset/CXRAY/Landmarks_3_10/images"
mask_base = "../Dataset/CXRAY/Landmarks_3_10/masks"

i = 1

image_reference = []

nnUNet_image_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset123_CXRAY/imagesTs"
nnUNet_mask_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset123_CXRAY/labelsTs"
os.makedirs(nnUNet_image_base, exist_ok=True)
os.makedirs(nnUNet_mask_base, exist_ok=True)

for row in test.iterrows():
    image_path = os.path.join(image_base, row[1][0])
    mask_path = os.path.join(mask_base, row[1][0])

    image_new_name = "image_%s_0000.png" % i
    mask_new_name = "image_%s.png" % i

    image_reference.append("image_%s" % i)

    # copy image and mask
    image_dest = os.path.join(nnUNet_image_base, image_new_name)
    mask_dest = os.path.join(nnUNet_mask_base, mask_new_name)

    os.system("cp %s %s" % (image_path, image_dest))
    os.system("cp %s %s" % (mask_path, mask_dest))

    i += 1

test = test.assign(image_reference=image_reference)
test.to_csv("../cxray_test_reference.csv", index=False)

In [9]:
test

,0,image_reference
0,CHNCXR_0157_0.png,image_1
1,CHNCXR_0604_1.png,image_2
2,CHNCXR_0600_1.png,image_3
3,CHNCXR_0613_1.png,image_4
4,CHNCXR_0279_0.png,image_5
...,...,...
176,JPCNN057.png,image_177
177,JPCLN114.png,image_178
178,JPCLN068.png,image_179
179,JPCNN008.png,image_180


In [30]:
from medpy.metric.binary import dc, hd, __surface_distances
import numpy as np
import cv2 
import pandas as pd
import os

def calculate_metrics(pred, gt):    
    dc_val = dc(pred, gt)    
    hd_val = hd(pred, gt, voxelspacing=(1, 1))
    d1 = __surface_distances(pred, gt, voxelspacing=(1, 1))
    d2 = __surface_distances(gt, pred, voxelspacing=(1, 1))
    assd_val = np.mean(np.concatenate((d1, d2)))
    return dc_val, hd_val, assd_val

test = pd.read_csv("../cxray_test_reference.csv")

organs = [1,2,3]
organ_names = ['Lungs', 'Heart', 'Clavicles']

nnUNet_pred_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_results/Dataset123_CXRAY/test"
results = []

nnUNet_image_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset123_CXRAY/imagesTs"
nnUNet_mask_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset123_CXRAY/labelsTs"

test['label_path'] = test['image_reference'].apply(lambda x: os.path.join(nnUNet_mask_base, x + ".png"))
test['pred_path'] = test['image_reference'].apply(lambda x: os.path.join(nnUNet_pred_base, x + ".png"))

for row in test.iterrows():
    GT = cv2.imread(row[1]['label_path'], cv2.IMREAD_UNCHANGED)
    mask = cv2.imread(row[1]['pred_path'], cv2.IMREAD_UNCHANGED)
    
    name = row[1][0]
    
    if name.startswith('MCU'):
        dataset = "Montgomery"
    elif name.startswith('JPC'):
        dataset = "JSRT"
    elif name.startswith('CHN'):
        dataset = "Shenzhen"
    else:
        dataset = "Padchest"
    
    for organ_num, organ_name in zip(organs, organ_names):
        gt_organ = (GT == int(organ_num)).astype(np.uint8)
        if gt_organ.sum() == 0:
            continue
        dc_val, hd_val, assd_val = calculate_metrics(mask == int(organ_num), gt_organ)
        
        results.append({
            "image": name,
            "dataset": dataset,
            "organ": organ_name,
            "dc": dc_val,
            "hd": hd_val,
            "assd": assd_val,
        })
        
results = pd.DataFrame(results)
results.to_csv("../Results/cxray_nnUNet_results.csv", index=False)



/tmp/ipykernel_3975977/1779619114.py:33: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  name = row[1][0]


In [31]:
results = pd.read_csv("../Results/cxray_nnUNet_results.csv")

In [32]:
# print mean and std for organ per dataset
for organ in organ_names:
    for dataset in ["Shenzhen", "Montgomery", "Padchest", "JSRT"]:
        print(f"Organ: {organ}, Dataset: {dataset}")
        df_subset = results[(results['organ'] == organ) & (results['dataset'] == dataset)]
        
        for metric in ['dc', 'hd', 'assd']:
            print(f" Metric: {metric}")
            mean_value = df_subset[metric].mean()
            std_value = df_subset[metric].std()
            mean_value = round(mean_value, 3)
            std_value = round(std_value, 3)
            print(f"  nnUNet: {mean_value:.3f} ± {std_value:.3f}")

Organ: Lungs, Dataset: Shenzhen
 Metric: dc
  nnUNet: 0.969 ± 0.014
 Metric: hd
  nnUNet: 33.012 ± 23.909
 Metric: assd
  nnUNet: 4.603 ± 1.993
Organ: Lungs, Dataset: Montgomery
 Metric: dc
  nnUNet: 0.982 ± 0.009
 Metric: hd
  nnUNet: 23.976 ± 19.805
 Metric: assd
  nnUNet: 2.811 ± 1.397
Organ: Lungs, Dataset: Padchest
 Metric: dc
  nnUNet: 0.965 ± 0.017
 Metric: hd
  nnUNet: 38.530 ± 20.616
 Metric: assd
  nnUNet: 4.687 ± 2.311
Organ: Lungs, Dataset: JSRT
 Metric: dc
  nnUNet: 0.979 ± 0.009
 Metric: hd
  nnUNet: 35.572 ± 25.931
 Metric: assd
  nnUNet: 2.824 ± 1.223
Organ: Heart, Dataset: Shenzhen
 Metric: dc
  nnUNet: nan ± nan
 Metric: hd
  nnUNet: nan ± nan
 Metric: assd
  nnUNet: nan ± nan
Organ: Heart, Dataset: Montgomery
 Metric: dc
  nnUNet: nan ± nan
 Metric: hd
  nnUNet: nan ± nan
 Metric: assd
  nnUNet: nan ± nan
Organ: Heart, Dataset: Padchest
 Metric: dc
  nnUNet: 0.947 ± 0.016
 Metric: hd
  nnUNet: 31.784 ± 12.136
 Metric: assd
  nnUNet: 9.372 ± 2.857
Organ: Heart, Datase